# LexID · RAG legal dengan sitasi

Kendrick Filbert. Empat PDF tugas, hybrid retrieval, parent-child, HyDE, reranker, DuckDuckGo dan input/Markdown. **Pengujian lokal hanya retrieval; generation menunggu model mahasiswa.**

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("LEGAL_RAG_ROOT", Path.cwd())).resolve()
assert (ROOT / "src/legal_rag").is_dir(), "Set LEGAL_RAG_ROOT to the extracted project folder"
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
from legal_rag.config import settings, read_json, write_json
cfg = settings()
RUN_TRAINING = os.getenv("LEGAL_RAG_TRAIN") == "1"
if RUN_TRAINING:
    import unsloth  # Must precede transformers / sentence_transformers in this kernel.
print("Nama: Kendrick Filbert")
print("Mode:", "GPU training enabled" if RUN_TRAINING else "CPU verification; GPU training NOT executed")
def cli(*args):
    subprocess.run([sys.executable, "-m", "legal_rag.cli", *args], cwd=ROOT, check=True)


## 1. Korpus lengkap, metadata dan OCR
Nomor halaman adalah halaman PDF 1-based. Tabel sideways PP5 memerlukan OCR; cache menyimpan confidence dan koordinat. Regulasi tugas adalah korpus historis, bukan verifikasi status hukum terkini.

In [ ]:
from legal_rag.corpus import load_pages, make_chunks
if not (ROOT / "data/processed/pages.jsonl").exists():
    cli("extract")
if not (ROOT / "artifacts/reports/ocr.json").exists():
    cli("ocr")
pages = load_pages()
parents, children = make_chunks(pages)
print(json.dumps(read_json(ROOT / "data/processed/manifest.json"), indent=2))
print("OCR report:", read_json(ROOT / "artifacts/reports/ocr.json"))
print("Pages, parents, children:", len(pages), len(parents), len(children))
assert len({p["metadata"]["doc_id"] for p in pages}) == 4
print("Child sample metadata:", children[0].metadata)
print("Chunk config:", {k:cfg[k] for k in ["child_size", "child_overlap", "parent_size", "parent_overlap"]})

## 2. Embedding open-source dan FAISS lokal
E5 memakai prefix `query:` / `passage:` dan normalized vectors untuk inner product. BM25/RRF berbobot 0.45/0.55; metadata difilter sebelum perankingan. Setidaknya lima parent kandidat diambil sebelum cross-encoder Top-3.

In [ ]:
import torch
torch.set_num_threads(4)
from legal_rag.retrieval import HybridRetriever, Reranker
if (ROOT / "artifacts/index/children.faiss").exists():
    retriever = HybridRetriever.load()
else:
    retriever = HybridRetriever.build(parents, children)
reranker = Reranker()
print("Embedding:", cfg["embedding_model"])
print("Reranker:", cfg["reranker_model"])
print("FAISS vectors:", retriever.index.ntotal)

In [ ]:
query = "Apakah pekerja yang lembur harus dibayar upah kerja lembur?"
retrieved = retriever.retrieve(query)
print("Candidates before reranking:", len(retrieved))
assert len(retrieved) >= 5
for i, row in enumerate(retrieved[:5], 1):
    print(f"\n--- Dokumen {i} ---")
    print(row["chunk"].metadata["citation"])
    print(row["chunk"].text[:1800])
ranked = reranker.rerank(query, retrieved)
for row in ranked:
    print(row["chunk"].metadata["citation"], "raw logit", row["relevance_logit"], "sigmoid score", row["relevance_score"])

In [ ]:
filtered = retriever.retrieve("uang kompensasi PKWT", filters={"doc_id": "PP_35_2021"})
assert all(row["chunk"].metadata["doc_id"] == "PP_35_2021" for row in filtered)
print("Filtered candidates:", len(filtered))
print([row["chunk"].metadata["citation"] for row in filtered[:5]])
from legal_rag.evaluation import retrieval_evaluation
evaluation = retrieval_evaluation(retriever, reranker)
print(json.dumps(evaluation, ensure_ascii=False, indent=2))

## 3. Model hasil SFT+GRPO mahasiswa dan dua HyDE
Generator hanya menerima repo dalam receipt training mahasiswa. Hipotesis HyDE membantu pencarian dan tidak pernah menjadi sumber final. Skor threshold 0.45 provisional; kalibrasi memerlukan relevance labels held-out.

In [ ]:
from legal_rag.pipeline import LegalRAG, OVERTIME_CASE
rag = None
if RUN_TRAINING:
    from legal_rag.generation import StudentGenerator
    rag = LegalRAG(retriever, reranker, StudentGenerator(stage="grpo"))
    result = rag.ask(OVERTIME_CASE)
    print("HyDE outputs:", result.get("hypotheses"))
    print(json.dumps(result, ensure_ascii=False, indent=2))
    assert len(result["hypotheses"]) >= 2
    assert result["status"] == "answered", "Inspect sources/threshold before asserting success"
    assert "<think>" in result["answer"] and "</think>" in result["answer"]
    assert result["citation_ids_valid"]
    result["complete"] = True
    write_json(ROOT / "artifacts/reports/rag_end_to_end.json", result)
else:
    print("NOT RUN: HyDE, model reasoning summary, dan final answer belum tersedia karena training GPU belum dilakukan.")

## 4. Fallback DuckDuckGo ketika Top-1 rendah
Akses web memerlukan `LEGAL_RAG_WEB=1` dan kueri publik eksplisit. Jika threshold dilampaui, sumber lokal tetap dipakai. Jika skor rendah, sumber lokal dibuang dan hanya hasil resmi .go.id menjadi konteks. Snippet web tetap memerlukan pembacaan sumber lengkap. Jalur fixture diuji terpisah; fixture bukan inferensi LLM.

In [ ]:
from legal_rag.pipeline import should_fallback
print("Threshold:", cfg["rerank_threshold"], cfg["threshold_status"])
print("Actual top1 retrieval requires fallback:", should_fallback(ranked, cfg["rerank_threshold"]))
if rag is not None and os.getenv("LEGAL_RAG_WEB") == "1":
    public_query = "peraturan perundang-undangan perlindungan data pribadi Indonesia"
    web_demo = rag.ask(public_query, filters={"doc_id": "PP_35_2021"}, public_web_query=public_query)
    print(json.dumps(web_demo, ensure_ascii=False, indent=2))
    write_json(ROOT / "artifacts/reports/web_fallback.json", web_demo)
else:
    print("Live web fallback not executed: enable web and complete student model first.")

## 5. Antarmuka wajib: input() + IPython.display.Markdown
Pada eksekusi otomatis, pertanyaan contoh dan `keluar` diinjeksikan ke input() agar notebook selesai tanpa menunggu terminal. Jawaban tetap dihasilkan model nyata. Jalankan sel berikutnya secara manual untuk bertanya bebas.

In [ ]:
from legal_rag.interfaces import interactive_loop
if rag is not None:
    from unittest.mock import patch
    # Only input is scripted; model inference is real.
    with patch("builtins.input", side_effect=[OVERTIME_CASE, "keluar"]):
        interactive_loop(rag)
    write_json(ROOT / "artifacts/reports/rag_interface.json", {"complete": True, "interface": "input()+IPython.display.Markdown", "question": OVERTIME_CASE})
else:
    print("Interface inference NOT RUN: no trained student model. Function implementation:")
    import inspect
    print(inspect.getsource(interactive_loop))

In [ ]:
if rag is not None and os.getenv("LEGAL_RAG_INTERACTIVE") == "1":
    interactive_loop(rag)
else:
    print("Set LEGAL_RAG_INTERACTIVE=1 for an unscripted interactive session.")

Opsional portfolio: `python -m legal_rag.cli ui` membuka Gradio pada localhost. Ini melengkapi loop wajib, tidak menggantikannya.

In [ ]:
from subprocess import run
check = run([sys.executable, "scripts/validate_submission.py"], cwd=ROOT, text=True, capture_output=True)
print(check.stdout)
print("Validator exit code:", check.returncode)
print("Draf belum layak dikumpulkan" if check.returncode else "Bukti struktural lengkap; tinjau kualitas sebelum dikumpulkan")